In [22]:
from __future__ import annotations

from typing import TypedDict, Optional, List, Dict, Literal, Annotated
from datetime import datetime
import json

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from typing import Dict


In [42]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

## 전체 state

In [ ]:
# from typing import TypedDict

# class PatentState(TypedDict):
#     summary_data: ConsultationData  # 요약 노드가 쓰는 방
#     claims_data: ClaimResult         # 💡 청구항 노드가 쓰는 방
#     examiner_data: ExaminerResult   # 심사관 노드가 쓰는 방

## 요약 state

In [ ]:
# class Element(TypedDict):
#     element_id: int          # int: 몇 번째 구성요소인지
#     description: str         # str: 설명
#     parent_id: Optional[int] # 세부 구성요소일 경우, 상위 구성요소의 id

In [ ]:
# class ConsultationData(TypedDict):
#     problems: List[str]      # 해결하고자 하는 과제 (기존 기술 문제점)
#     elements: List[Element]  # Element 객체(Dict)들이 담긴 리스트 구조
#     effects: List[str]       # 효과
#     user_confirmed: bool     # 요약 정리 동의 여부

In [52]:
sample_consultation : ConsultationData = {
    "problems": [
        "기존의 특허 초안 작성 프로세스는 변리사의 수작업에 의존하여 많은 시간과 비용이 소모됨.",
        "일반적인 LLM을 사용할 경우 특허법상의 기재불량(명확성 위배, 다다중 인용 등)을 걸러내지 못해 실무 적용이 어려움."
    ],
    "elements": [
        {
            "element_id": 1,
            "description": "발명가로부터 발명의 목적, 핵심 구성, 기대 효과를 입력받는 유저 인터페이스 및 입력 데이터 파싱 모듈",
            "parent_id": None  # 최상위 구성요소이므로 None
        },
        {
            "element_id": 2,
            "description": "유저의 명령 의도를 분석하고 적절한 서브 에이전트를 호출하여 제어하는 마스터 매니저 에이전트",
            "parent_id": None  # 최상위 구성요소이므로 None
        },
        {
            "element_id": 3,
            "description": "마스터 매니저 에이전트 내에서 입력된 의도에 따라 선행기술조사, 청구항 생성, 명세서 생성을 분기하는 라우팅 모듈",
            "parent_id": 2     # element_id 2번(마스터 매니저) 하위의 세부 구성요소
        },
        {
            "element_id": 4,
            "description": "생성된 청구항의 특허법 제42조 제4항 제2호(명확성) 및 시행령 제4조 제4항(다다중 인용) 위배 여부를 실시간 심사하는 검증 모듈",
            "parent_id": 2     # element_id 2번(마스터 매니저) 하위의 세부 구성요소
        }
    ],
    "effects": [
        "특허 명세서 초안 작성에 소요되는 시간을 기존 대비 50% 이상 단축 가능.",
        "법적 기재 요건을 실시간으로 검증함으로써 초안의 결함율을 최소화하고 변리사의 검토 효율을 극대화함."
    ],
    "user_confirmed": False  # 아직 최초 요약 단계이므로 발명가의 컨펌을 기다리는 상태
}

## 청구항 state

In [ ]:
# class ClaimItem(TypedDict):
#     claim_no: int
#     is_dependent: bool
#     cited_claim_no: List[int]  # 인용항이 없으면 빈 리스트 [] 반환
#     category: Literal["방법", "시스템", "CRM"]
#     content: str


In [ ]:
# class ClaimResult(TypedDict):
#     claims: List[ClaimItem]

In [47]:
claims_data: ClaimResult = {
    "claims": [
        {
            "claim_no": 1,
            "is_dependent": False,
            "cited_claim_no": [],  # 독립항이므로 빈 리스트
            "category": "방법",
            "content": "인공지능 기반의 특허 문서 자동 초안 작성 방법으로서, 데이터를 수집하는 단계와, ...를 특징으로 하는 방법."
        },
        {
            "claim_no": 2,
            "is_dependent": True,
            "cited_claim_no": [1],  # 1항을 인용하는 단일 종속항
            "category": "방법",
            "content": "제1항에 있어서, 상기 데이터를 수집하는 단계는, 사용자가 입력한 ...를 포함하는 방법."
        },
        {
            "claim_no": 3,
            "is_dependent": True,
            "cited_claim_no": [1, 2],  # 1항 또는 2항을 인용하는 다중 인용항 (질문하신 형태!)
            "category": "방법",
            "content": "제1항 또는 제2항에 있어서, 상기 초안을 검토하는 단계를 더 포함하는 방법."
        },
        {
            "claim_no": 4,
            "is_dependent": False,
            "cited_claim_no": [],
            "category": "시스템",
            "content": "인공지능 기반의 특허 문서 자동 초안 작성 시스템으로서, 프로세서; 및 상기 프로세서에 연결된 메모리를 포함하고, ...시스템."
        }
    ]
}

## 심사관 state

In [ ]:
# class RejectionDetail(TypedDict):
#     claims: List[int]        # 거절 이유에 해당하는 청구항 번호들 (예: [1, 3, 4])
#     reason_text: str         # 명확성 요건 위배 이유 구체적 기술 (또는 기존 RejectionReason 객체)

In [ ]:
# class ExaminerResult(TypedDict):
#     is_approved: bool
#     rejections: List[RejectionDetail]  # 청구항별 거절 이유 매핑 리스트
#     revision_count: int                # 최대 2번 루프 제어용

In [ ]:
# # LLM 심사 결과가 반영된 State 예시
# result: ExaminerResult = {
#     "is_approved": False,
#     "rejections": [
#         {
#             "claims": [1, 3, 4],
#             "reason_text": "청구범위가 불분명하여 발명이 명확하게 파악되지 않음 (구성요소 A의 유기적 결합 관계 모호)",
#             "basis_law": "특허법 제42조제4항제2호"
#         },
#         {
#             "claims": [2, 5],
#             "reason_text": "단어 '상기 ...'의 선행 명사가 존재하지 않아 청구항의 의미가 불명확함",
#             "basis_law": "특허법 제42조제4항제2호"
#         }
#     ],
#     "revision_count": 0
# }

## 청구항 agent의 run

In [ ]:
from typing import Dict, List

# 청구항 생성을 위한 프롬프트나 내부 로직에서 활용할 규칙 기반 매핑 함수
def generate_claims_structure(consultation: ConsultationData) -> ClaimResult:
    """
    발명의 요약 데이터(ConsultationData)를 분석하여
    특허법적 계층 구조를 가진 청구항 구조(ClaimResult)의 뼈대를 생성합니다.
    """
    elements = consultation["elements"]
    problems = consultation["problems"]
    effects = consultation["effects"]
    
    # 1. 빠른 조회를 위해 element_id별로 데이터를 매핑할 딕셔너리 생성
    element_map: Dict[int, Element] = {el["element_id"]: el for el in elements}
    
    # 2. 청구항 번호 매핑을 위한 딕셔너리 (element_id -> claim_no)
    # 어떤 구성요소가 몇 번 청구항으로 만들어졌는지 추적합니다.
    el_to_claim_map: Dict[int, int] = {}
    
    claims_list: List[ClaimItem] = []
    current_claim_no = 1

    # =================================================================
    # [STEP 1] 독립항(Independent Claim) 생성
    # parent_id가 없는 최상위 핵심 구성요소들을 찾아 각각 독립항으로 빌드합니다.
    # =================================================================
    top_elements = [el for el in elements if el["parent_id"] is None]
    
    for el in top_elements:
        el_id = el["element_id"]
        el_to_claim_map[el_id] = current_claim_no
        
        # 독립항 category 판단 (요약문 텍스트에서 힌트를 얻거나 기본값 지정)
        category = "시스템" if "시스템" in el["description"] else "방법"
        
        # 발명의 목적(problems)과 효과(effects)를 독립항 콘텐트에 유기적으로 녹여냄
        content = (
            f"[해결과제: {problems[0] if problems else ''}]를 해결하기 위한 {el['description']}로서, "
            f"상기 {el['description']}의 세부 메커니즘을 포함하여 [기대효과: {effects[0] if effects else ''}]를 특징으로 하는 {category}."
        )
        
        independent_claim: ClaimItem = {
            "claim_no": current_claim_no,
            "is_dependent": False,
            "cited_claim_no": [],  # 독립항이므로 빈 리스트
            "category": category,
            "content": content
        }
        claims_list.append(independent_claim)
        current_claim_no += 1

    # =================================================================
    # [STEP 2] 종속항(Dependent Claim) 생성
    # parent_id가 존재하는 세부 구성요소들을 찾아 상위 청구항을 인용하는 종속항으로 빌드합니다.
    # =================================================================
    sub_elements = [el for el in elements if el["parent_id"] is not None]
    
    for el in sub_elements:
        el_id = el["element_id"]
        parent_id = el["parent_id"]
        
        # 부모 구성요소가 상기 독립항 단계에서 몇 번 청구항으로 선언되었는지 확인
        parent_claim_no = el_to_claim_map.get(parent_id)
        
        if parent_claim_no is not None:
            el_to_claim_map[el_id] = current_claim_no
            parent_claim = next((c for c in claims_list if c["claim_no"] == parent_claim_no), None)
            
            # 종속항은 부모 청구항의 category를 엄격하게 추종해야 함 (Mismatched Category 방지)
            category = parent_claim["category"]
            
            # 특허 정식 기재 양식 적용 ("제X항에 있어서, 상기...")
            content = (
                f"제{parent_claim_no}항에 있어서, "
                f"상기 {parent_map_desc(element_map, parent_id)}는, "
                f"{el['description']}을(를) 더 포함하는 것을 특징으로 하는 {category}."
            )
            
            dependent_claim: ClaimItem = {
                "claim_no": current_claim_no,
                "is_dependent": True,
                "cited_claim_no": [parent_claim_no],  # 부모 청구항 번호 자동 인용
                "category": category,
                "content": content
            }
            claims_list.append(dependent_claim)
            current_claim_no += 1

    return {"claims": claims_list}

# def parent_map_desc(element_map: Dict[int, Element], parent_id: int) -> str:
#     """부모 구성요소의 명칭을 정제해서 가져오는 헬퍼 함수"""
#     full_desc = element_map[parent_id]["description"]
#     # 프롬프트나 로직에서 명칭만 싹 잘라 쓰기 위해 앞글자 10자만 예시로 파싱 (실무에선 체계적 명사 추출 필요)
#     return full_desc.split("하는")[-1].strip() if "하는" in full_desc else full_desc[:15]

In [32]:
import json
from typing import Dict
from langchain_openai import ChatOpenAI

class ClaimGenerationAgent:
    def __init__(self):
        # 청구항의 디테일한 표현력을 위해 gpt-5-mini 계열을 권장하며, 
        # 구조적 일관성을 위해 temperature는 낮게(0.1 ~ 0.2) 잡는 것이 좋습니다.
        self.llm = ChatOpenAI(model="gpt-5-mini", temperature=0.2)

    def run(self, state: PatentState) -> Dict:
        """
        요약 데이터를 바탕으로 특허 청구범위를 생성하는 노드 함수
        """
        print("[CL Agent] 구조화된 출력을 활용한 청구항 생성 시작...")
        
        # 1. State에서 이전 노드가 저장한 요약 데이터 꺼내기
        consultation_data = state.get("summary_data")
        if not consultation_data:
            raise ValueError("State에 summary_data가 존재하지 않습니다. 요약 노드를 먼저 확인하세요.")
        
        # 2. 변리사님이 설계한 규칙 기반의 청구항 '뼈대(틀)' 자동 빌드
        # 이 가이드라인이 LLM에게 기준점 역할을 해줍니다.
        base_claim_structure = generate_claims_structure(consultation_data)
        
        # 3. LLM에게 프롬프트 주입
        # 구조를 깨지 말고 '텍스트(content)'만 특허 실무 양식에 맞게 다듬으라고 명확히 지시합니다.
        system_prompt = (
            "당신은 대한민국의 베테랑 특허출원 전문 변리사입니다.\n"
            "제공된 청구항 JSON 구조의 'claim_no', 'is_dependent', 'cited_claim_no', 'category' 데이터는 "
            "특허법적 인용 관계를 고려하여 철저히 계산된 뼈대이므로 절대로 수정하거나 가공하지 마십시오.\n\n"
            "당신의 임무는 오직 각 청구항의 'content' 문자열을 대한민국 특허 실무 규정에 맞는 "
            "세련되고 유기적인 청구범위 문체(예: ~을 특징으로 하는 방법.)로 확장하고 다듬는 것입니다."
        )
        
        user_prompt = (
            f"아래의 청구항 뼈대 구조를 바탕으로 content 영역의 문장을 변리사답게 다듬어 주세요.\n"
            f"입력 데이터: {json.dumps(base_claim_structure, ensure_ascii=False)}"
        )
        
        # 4. 💡 .with_structured_output(ClaimResult) 적용
        # 이렇게 선언하면 LLM은 무조건 우리가 정의한 ClaimResult 스키마에 맞춰 JSON을 생성합니다.
        structured_llm = self.llm.with_structured_output(ClaimResult)
        
        # 5. LLM 호출 및 결과 수신 (결과는 문자열이 아니라 곧바로 '파이썬 dict'로 들어옵니다)
        refined_claims: ClaimResult = structured_llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ])
        
        # 6. LangGraph State의 claims_data 영역에 write 하기 위해 딕셔너리 반환
        return {
            "claims_data": refined_claims
        }

In [43]:
claim_agent = ClaimGenerationAgent()

In [53]:
mock_state = {
    "summary_data": sample_consultation
}

In [56]:
claim_example = claim_agent.run(mock_state)

[CL Agent] 구조화된 출력을 활용한 청구항 생성 시작...


In [70]:
print(json.dumps(claim_example, indent=4, ensure_ascii=False))

{
    "claims_data": {
        "claims": [
            {
                "claim_no": 1,
                "is_dependent": false,
                "cited_claim_no": [],
                "category": "방법",
                "content": "발명의 해결과제는 기존의 특허 초안 작성 프로세스가 변리사의 수작업에 의존하여 많은 시간과 비용이 소모되는 점을 해결하는 것에 있으며, 이에 따라 발명가로부터 발명의 목적, 핵심 구성 및 기대효과를 입력받는 유저 인터페이스 및 입력 데이터 파싱 모듈을 제공하는 방법으로서, 상기 유저 인터페이스 및 입력 데이터 파싱 모듈의 세부 메커니즘을 포함하여 특허 명세서 초안 작성에 소요되는 시간을 종래 대비 50% 이상 단축할 수 있음을 특징으로 하는 방법."
            },
            {
                "claim_no": 2,
                "is_dependent": false,
                "cited_claim_no": [],
                "category": "방법",
                "content": "발명의 해결과제는 기존의 특허 초안 작성 프로세스가 변리사의 수작업에 의존하여 많은 시간과 비용이 소모되는 점을 해결하는 것에 있으며, 이에 따라 유저의 명령 의도를 분석하고 적절한 서브 에이전트를 호출하여 제어하는 마스터 매니저 에이전트를 제공하는 방법으로서, 상기 유저의 명령 의도를 분석하고 적절한 서브 에이전트를 호출하여 제어하는 마스터 매니저 에이전트의 세부 메커니즘을 포함하여 특허 명세서 초안 작성에 소요되는 시간을 종래 대비 50% 이상 단축할 수 있음을 특징으로 하는 방법."
            },
            {
               

## 심사관 agent의 run

In [ ]:
# import json
# from typing import Dict
# from langchain_openai import ChatOpenAI

# class ExaminerAgent:
#     def __init__(self):
#         # 💡 [핵심] 변리사님이 수집한 데이터로 '파인튜닝한 모델의 ID'를 여기에 세팅합니다.
#         # 구조적 답변을 엄격히 통제하기 위해 temperature는 반드시 0으로 설정합니다.
#         self.ft_model = ChatOpenAI(
#             model="ft:gpt-4o-mini:your-patent-firm:examiner-critic-v1", 
#             temperature=0
#         )

#     def _format_claims_to_text(self, claims_result: ClaimResult) -> str:
#         """LLM 심사관이 읽기 좋게 ClaimResult 구조를 텍스트 전문으로 변환하는 헬퍼 함수"""
#         text_lines = []
#         for c in claims_result["claims"]:
#             prefix = "[종속항]" if c["is_dependent"] else "[독립항]"
#             text_lines.append(f"{prefix} 청구항 {c['claim_no']} ({c['category']})")
#             if c["is_dependent"]:
#                 text_lines.append(f"인용항 번호: {c['cited_claim_no']}")
#             text_lines.append(f"내용: {c['content']}\n")
#         return "\n".join(text_lines)

#     def run(self, state: PatentState) -> Dict:
#         """
#         심사관 에이전트 핵심 실행 함수 (run)
#         1. CL 노드가 작성한 claims_data를 가져와 텍스트로 파싱합니다.
#         2. 파인튜닝된 모델을 호출하여 오직 '명확성 요건'만 심사합니다.
#         3. 결과를 우리가 정의한 ExaminerResult 구조로 변환하여 State를 업데이트합니다.
#         """
#         print("[Critic Agent] 파인튜닝 모델 기반 특허 심사 시작...")
        
#         # 1. State에서 이전 노드(CL)가 생성한 청구항 데이터 가져오기
#         claims_data: ClaimResult = state.get("claims_data")
#         if not claims_data:
#             raise ValueError("State에 claims_data가 존재하지 않습니다. 청구항 생성 노드를 먼저 확인하세요.")
        
#         # 2. 파인튜닝 모델에게 주입할 유저 프롬프트(청구범위 전문) 생성
#         claims_text_for_review = self._format_claims_to_text(claims_data)
        
#         # 3. 파인튜닝 학습 시 사용했던 System Prompt와 완벽히 싱크를 맞춥니다.
#         system_prompt = (
#             "당신은 대한민국의 베테랑 특허심사관입니다. 입력된 청구범위를 검토하여 "
#             "특허법 제42조 제4항 제2호(명확성 요건) 및 특허법 시행령 제4조 제4항(다다중 인용 금지) "
#             "위배 여부를 심사하고, 결과를 지정된 ExaminerResult JSON 형식으로만 반환하세요."
#         )
        
#         # 4. 💡 .with_structured_output 적용
#         # 앞서 설계해둔 ExaminerResult 구조를 강제하여 파인튜닝 모델의 출력을 안전하게 가둡니다.
#         structured_ft_model = self.ft_model.with_structured_output(ExaminerResult)
        
#         # 5. 파인튜닝 모델 호출 (결과는 곧바로 파이썬 dict로 수신)
#         examination_output: ExaminerResult = structured_ft_model.invoke([
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": claims_text_for_review}
#         ])
        
#         # 6. 루프 제어를 위한 안전장치 (기존 revision_count가 있다면 1 증가, 없으면 0으로 시작)
#         # 이 카운트를 보고 LangGraph 라우터가 '보정 노드'로 보낼지 'Done'으로 보낼지 결정합니다.
#         current_count = state.get("examiner_data", {}).get("revision_count", 0)
        
#         # 만약 심사 결과 거절이유가 나왔다면 카운트를 올립니다.
#         if not examination_output["is_approved"]:
#             examination_output["revision_count"] = current_count + 1
#         else:
#             examination_output["revision_count"] = current_count

#         # 7. LangGraph State의 examiner_data 영역으로 write
#         return {
#             "examiner_data": examination_output
#         }

In [ ]:
# import json
# from typing import Dict
# from langchain_openai import ChatOpenAI

# class ExaminerAgent:
#     def __init__(self):
#         # 💡 향후 gpt-5-mini 등 원하시는 최신 상용 모델 ID로 교체하시면 됩니다.
#         # 심사의 정확성과 일관성을 위해 temperature는 0으로 철저히 고정합니다.
#         self.llm = ChatOpenAI(
#             model="gpt-5-mini",  # 현재는 테스트용으로 gpt-4o 지정 (향후 gpt-5-mini로 변경 가능)
#             temperature=0
#         )

#     def _format_claims_to_text(self, claims_result: ClaimResult) -> str:
#         """LLM 심사관이 청구항 간의 구조를 한눈에 파악할 수 있도록 텍스트 전문으로 정제"""
#         text_lines = []
#         for c in claims_result["claims"]:
#             prefix = "[종속항]" if c["is_dependent"] else "[독립항]"
#             text_lines.append(f"{prefix} 청구항 {c['claim_no']} ({c['category']})")
#             if c["is_dependent"]:
#                 text_lines.append(f"인용 대상 청구항 번호들: {c['cited_claim_no']}")
#             text_lines.append(f"내용: {c['content']}\n")
#         return "\n".join(text_lines)

#     def run(self, state: PatentState) -> Dict:
#         """
#         일반 LLM 기반 심사관 에이전트 핵심 실행 함수 (run)
#         """
#         print("[Critic Agent] 일반 LLM 기반 특허 심사 시작...")
        
#         # 1. State에서 검사할 청구항 데이터 가져오기
#         claims_data: ClaimResult = state.get("claims_data")
#         if not claims_data:
#             raise ValueError("State에 claims_data가 존재하지 않습니다. 청구항 생성 노드를 먼저 확인하세요.")
        
#         # 2. 청구범위 전문 텍스트 변환
#         claims_text_for_review = self._format_claims_to_text(claims_data)
        
#         # 3. 일반 LLM이 똑똑하게 판단할 수 있도록 '특허법적 심사 기준'을 명확히 주입 (Few-Shot 역할)
#         system_prompt = (
#             "당신은 대한민국 특허청의 엄격한 베테랑 특허심사관입니다.\n"
#             "입력된 청구범위를 검토하여 다음 '2가지 법적 요건'만 집중적으로 심사해 주세요.\n\n"
            
#             "=== 심사 기준 ===\n"
#             "1. 특허법 제42조 제4항 제2호 (명확성 요건):\n"
#             "   - 청구항에 기재된 발명이 모호하거나 불명확하여 발명의 구성을 파악할 수 없는 경우 거절.\n"
#             "   - 특히, 앞서 선언되지 않은 구성요소를 '상기 [구성요소]'라고 인용하는 경우(선행 명사 부존재) 기재불량으로 적발할 것.\n"
#             "2. 특허법 시행령 제4조 제4항 (다중인용항의 다중인용항 인용 금지):\n"
#             "   - 2개 이상의 청구항을 인용하는 청구항을 '다중인용항'이라 합니다.\n"
#             "   - 어떤 청구항이 인용하는 대상 중에 '이미 다중인용항인 청구항'이 단 하나라도 포함되어 있다면, 이는 '다다중 인용'으로 무조건 거절해야 합니다.\n\n"
            
#             "=== 반환 형식 ===\n"
#             "반드시 제공된 ExaminerResult 스키마 규격을 준수하여 응답하세요."
#         )
        
#         # 4. 구조화된 출력(Structured Output) 결합
#         structured_llm = self.llm.with_structured_output(ExaminerResult)
        
#         # 5. 모델 호출 및 결과 수신
#         examination_output: ExaminerResult = structured_llm.invoke([
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": f"이하의 청구범위를 엄격하게 심사하십시오:\n\n{claims_text_for_review}"}
#         ])
        
#         # 6. 루프 제어용 revision_count 안전장치 작동
#         current_count = state.get("examiner_data", {}).get("revision_count", 0)
#         if not examination_output["is_approved"]:
#             examination_output["revision_count"] = current_count + 1
#         else:
#             examination_output["revision_count"] = current_count

#         # 7. LangGraph State로 결과물 리턴
#         return {
#             "examiner_data": examination_output
#         }

In [61]:
examiner_agent = ExaminerAgent()

In [66]:
mock_state_for_critic = {
    "claims_data": claim_example["claims_data"], # 아까 generate_claims_structure 함수가 뱉은 결과물
    "examiner_data": {"revision_count": 0} # 초기 카운트 세팅
}

In [67]:
final_examination = examiner_agent.run(mock_state_for_critic)
print(json.dumps(final_examination, indent=4, ensure_ascii=False))

[Critic Agent] 일반 LLM 기반 특허 심사 시작...
{
    "examiner_data": {
        "is_approved": true,
        "rejections": [],
        "revision_count": 0
    }
}


In [68]:
# 일부러 법적 결함을 심어놓은 엉터리 청구항 데이터
bad_claims_data: ClaimResult = {
    "claims": [
        {
            "claim_no": 1,
            "is_dependent": False,
            "cited_claim_no": [],
            "category": "방법",
            "content": "발명가로부터 아이디어를 입력받아 요약하는 데이터 파싱 방법."
        },
        {
            "claim_no": 2,
            "is_dependent": True,
            "cited_claim_no": [1],
            "category": "방법",
            "content": "제1항에 있어서, 상기 요약하는 단계는 LLM을 이용하는 데이터 파싱 방법."
        },
        {
            "claim_no": 3,
            "is_dependent": True,
            "cited_claim_no": [1],
            "category": "방법",
            "content": "제1항에 있어서, 상기 벡터 데이터베이스는 결과를 임베딩하여 저장하는 것을 특징으로 하는 데이터 파싱 방법." 
            # ⚠️ 트랩 1 (명확성 위배): 1항과 2항 그 어디에도 '벡터 데이터베이스'를 선언한 적이 없는데 
            # 갑자기 '상기 벡터 데이터베이스'라고 인용하여 선행 명사가 부존재합니다.
        },
        {
            "claim_no": 4,
            "is_dependent": True,
            "cited_claim_no": [1, 2], # 1항 또는 2항을 인용하는 정상적인 '다중인용항'
            "category": "방법",
            "content": "제1항 또는 제2항에 있어서, 결과를 화면에 출력하는 단계를 더 포함하는 데이터 파싱 방법."
        },
        {
            "claim_no": 5,
            "is_dependent": True,
            "cited_claim_no": [3, 4], 
            # ⚠️ 트랩 2 (다다중 인용 위배): 인용하는 4번 항이 이미 '다중인용항([1, 2])'인데 
            # 이를 또 다른 항(3항)과 함께 다중 인용([3, 4])하여 시행령 제4조 제4항을 정면으로 위배합니다.
            "category": "방법",
            "content": "제3항 또는 제4항에 있어서, 알림을 전송하는 단계를 더 포함하는 데이터 파싱 방법."
        }
    ]
}

# 엉터리 데이터를 State에 주입
mock_state_for_critic = {
    "claims_data": bad_claims_data,
    "examiner_data": {"revision_count": 0}
}

# 심사관 출격!
final_examination = examiner_agent.run(mock_state_for_critic)
print(json.dumps(final_examination, indent=4, ensure_ascii=False))

[Critic Agent] 일반 LLM 기반 특허 심사 시작...
{
    "examiner_data": {
        "is_approved": false,
        "rejections": [
            {
                "claims": [
                    3
                ],
                "reason_text": "제3항은 '상기 벡터 데이터베이스'라고 기재하여 선행 명사(벡터 데이터베이스)가 존재함을 전제로 하고 있으나, 인용대상인 제1항 및 전체 기재에서 '벡터 데이터베이스'에 해당하는 구성요소가 명시적으로 개시되어 있지 않습니다. 즉, 선행 명사 부존재에 해당하여 청구항의 구성(특히 어느 구성요소를 가리키는지)을 파악할 수 없으므로 명확성 요건을 충족하지 못합니다. (예: '상기 벡터 데이터베이스'를 사용할 근거가 제1항에 없음)",
                "basis_law": "특허법 제42조 제4항 제2호 (명확성 요건)"
            },
            {
                "claims": [
                    5
                ],
                "reason_text": "제5항은 제3항 및 제4항을 다중인용(2개 이상 인용)하고 있습니다. 인용된 청구항들 중 제4항은 이미 제1항 및 제2항을 동시에 인용하는 '다중인용항'입니다. 시행령상 다중인용항을 인용대상에 포함한 다중인용은 '다다중 인용'에 해당하여 허용되지 않으므로 제5항은 거절됩니다. (구성상 또는 기재불량 문제와 무관하게 다중인용항 자체의 인용관계 위배로 거절)",
                "basis_law": "특허법 시행령 제4조 제4항 (다중인용항의 다중인용항 인용 금지)"
            }
        ],
        "revision_count": 1
    }
}


## 느낀점
- 클래스
- test 데이터셋(mock데이터) 별도 관리
- 로직은 다듬으면 된다.~ 